# **Pre-Trained Model 3 - DenseNet**

As a third pre-trained model we have choosen DenseNet's for its core idea — dense connectivity, where every layer receives feature
maps from all preceding layers, which we believe to be well-suited for our model. The dense connections encourage feature reuse, which helps the network
learn subtle dermoscopic patterns (pigment networks, vascular structures) with
fewer parameters than equivalent ResNet-style architectures.

The three variants of the model will be tested - 

### Imports

In [1]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

from model_utils import get_callbacks, plot_history, evaluate_model, plot_comparison

### Configuration

In [2]:
DENSENET_CONFIGS = [
    ("DenseNet121", keras.applications.DenseNet121, -50),
    ("DenseNet169", keras.applications.DenseNet169, -70),
    ("DenseNet201", keras.applications.DenseNet201, -85),
]

In [3]:
def build_densenet(application, unfreeze_from):
    """
    Builds a DenseNet model with a frozen pretrained base and a
    custom classifier head. The same head architecture is used for
    all three variants so results are directly comparable.

    Parameters
    ----------
    application : keras.applications class
        One of DenseNet121, DenseNet169, DenseNet201.
    unfreeze_from : int
        Negative index into base.layers — layers from this index
        onward are unfrozen in Phase 2.
    """
    base = application(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling=None
    )
    base.trainable = False

    inputs  = keras.Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.4)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(N_CLASSES)(x)

    return keras.Model(inputs, outputs), base, unfreeze_from

### Automated training Loop
Stores results from all three variants for the final comparison.


In [4]:
for label, application, unfreeze_from in DENSENET_CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Training {label}")
    print(f"{'='*60}\n")

    safe_label = label.replace(" ", "_").replace("—", "").replace("/", "")
    ckpt_phase1 = f"checkpoints/{safe_label}_phase1.weights.h5"
    ckpt_best   = f"checkpoints/{safe_label}_best.weights.h5"

    # ── Phase 1: head only ────────────────────────────────────────────────
    model, base, unfreeze_idx = build_densenet(application, unfreeze_from)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    print(f"Phase 1 — training head only ({label})")
    history_p1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        steps_per_epoch=STEPS_PER_EPOCH,
        class_weight=class_weights_dict,
        callbacks=get_callbacks(ckpt_phase1, patience_es=6, patience_lr=3)
    )
    plot_history(history_p1, f"{label} — Phase 1 (head only)")

    # ── Phase 2: fine-tune top layers ─────────────────────────────────────
    base.trainable = True
    for layer in base.layers[:unfreeze_idx]:
        layer.trainable = False

    trainable_count = sum(1 for w in model.trainable_weights)
    print(f"\nPhase 2 — fine-tuning top layers ({trainable_count} trainable tensors)")

    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"]
    )

    history_p2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        steps_per_epoch=STEPS_PER_EPOCH,
        class_weight=class_weights_dict,
        callbacks=get_callbacks(ckpt_best, patience_es=8, patience_lr=4)
    )
    plot_history(history_p2, f"{label} — Phase 2 (fine-tune)")

    # ── Evaluate ──────────────────────────────────────────────────────────
    results = evaluate_model(model, test_ds, label2idx, label)
    densenet_results[label]   = results
    densenet_histories[label] = {"phase1": history_p1, "phase2": history_p2}

    print(f"\n  {label} done — AUC: {results['macro_auc']:.4f} | "
          f"Mel recall: {results['mel_recall']:.4f}\n")



  Training DenseNet121

 6701056/29084464 ━━━━━━━━━━━━━━━━━━━━ 37s 2us/step

KeyboardInterrupt: 

### DENSENET INTERNAL COMPARISON 
Compare the three variants against each other before the global summary.

In [ ]:
plot_comparison(densenet_results)

### FULL PIPELINE COMPARISON

In [ ]:
all_results = {
    "A — Custom CNN": results_a,
    "B — EfficientNetB0": results_b,
    "C — MobileNetV2": results_c,
    **densenet_results,
}

plot_comparison(all_results)